In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

DataFrame[]

In [0]:
spark.sql("SHOW SCHEMAS").show(truncate=False)

+------------------+
|databaseName      |
+------------------+
|bronze            |
|default           |
|gold              |
|information_schema|
|silver            |
+------------------+



### Question 1: Which day(s) of the week sees the highest number of fraudulent transactions?

In [0]:
from pyspark.sql import functions as F

transactions_silver_tbl = spark.table("silver.transactions_silver")

fraud_by_day_of_week_df = (
    transactions_silver_tbl
    .filter(F.col("is_fraud") == True)
    .groupBy("day_of_week")
    .agg(F.count("*").alias("fraud_transaction_count"))
    .withColumn(
        "day_order",
        F.when(F.col("day_of_week") == "Monday", 1)
         .when(F.col("day_of_week") == "Tuesday", 2)
         .when(F.col("day_of_week") == "Wednesday", 3)
         .when(F.col("day_of_week") == "Thursday", 4)
         .when(F.col("day_of_week") == "Friday", 5)
         .when(F.col("day_of_week") == "Saturday", 6)
         .when(F.col("day_of_week") == "Sunday", 7)
    )
    .orderBy("day_order")
)

In [0]:
fraud_by_day_of_week_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold.fraud_by_day_of_week")

In [0]:
display(spark.table("gold.fraud_by_day_of_week"))

day_of_week,fraud_transaction_count,day_order
Monday,1747,1
Tuesday,2037,2
Wednesday,1102,3
Thursday,2082,4
Friday,2284,5
Saturday,1434,6
Sunday,2646,7


### Qestion 2: What is the trend of the fraud rate (fraudulent transactions divided by total) over the past month?

In [0]:
from pyspark.sql import functions as F

transactions_silver_tbl = spark.table("silver.transactions_silver")

fraud_rate_daily_df = (
    transactions_silver_tbl
    .groupBy("transaction_date")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum(F.when(F.col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transactions")
    )
    .withColumn(
        "fraud_rate",
        F.col("fraud_transactions") / F.col("total_transactions")
    )
    .orderBy("transaction_date")
)

display(fraud_rate_daily_df.limit(10))

transaction_date,total_transactions,fraud_transactions,fraud_rate
2010-01-01,3463,1,2.8876696505919725E-4
2010-01-02,2989,0,0.0
2010-01-03,3311,1,3.020235578375113E-4
2010-01-04,3244,2,6.165228113440197E-4
2010-01-05,3330,1,3.003003003003003E-4
2010-01-06,3365,0,0.0
2010-01-07,3346,2,5.977286312014345E-4
2010-01-08,3016,4,0.001326259946949602
2010-01-09,3102,1,3.223726627981947E-4
2010-01-10,3416,5,0.0014637002341920376


In [0]:
fraud_rate_daily_df.write.mode("overwrite").saveAsTable("gold.fraud_rate_daily")

In [0]:
display(spark.table("gold.fraud_rate_daily").limit(10))

transaction_date,total_transactions,fraud_transactions,fraud_rate
2010-01-01,3463,1,2.8876696505919725E-4
2010-01-02,2989,0,0.0
2010-01-03,3311,1,3.020235578375113E-4
2010-01-04,3244,2,6.165228113440197E-4
2010-01-05,3330,1,3.003003003003003E-4
2010-01-06,3365,0,0.0
2010-01-07,3346,2,5.977286312014345E-4
2010-01-08,3016,4,0.001326259946949602
2010-01-09,3102,1,3.223726627981947E-4
2010-01-10,3416,5,0.0014637002341920376


### Question 3: Which users have the largest number of flagged (is_fraud = true) transactions?

In [0]:
from pyspark.sql import functions as F

transactions_silver_tbl = spark.table("silver.transactions_silver")

top_users_by_fraud_count_df = (
    transactions_silver_tbl
    .filter(F.col("is_fraud") == True)
    .groupBy("client_id")
    .agg(F.count("*").alias("fraud_transaction_count"))
    .orderBy(F.desc("fraud_transaction_count"))
)

display(top_users_by_fraud_count_df.limit(10))

client_id,fraud_transaction_count
1102,58
209,52
27,45
155,44
1128,43
989,42
1851,42
1741,42
1649,41
359,39


In [0]:
top_users_by_fraud_count_df.write.mode("overwrite").saveAsTable("gold.top_users_by_fraud_count")

In [0]:
display(spark.table("gold.top_users_by_fraud_count").limit(10))

client_id,fraud_transaction_count
1102,58
209,52
27,45
155,44
1128,43
989,42
1851,42
1741,42
1649,41
359,39


In [0]:
from pyspark.sql import functions as F

top_10_users_by_fraud_count_df = (
    spark.table("gold.top_users_by_fraud_count")
    .orderBy(F.desc("fraud_transaction_count"))
    .limit(10)
)

top_10_users_by_fraud_count_df.write \
    .mode("overwrite") \
    .saveAsTable("gold.top_10_users_by_fraud_count")

### Question 4: Are there any users showing a sharp rise in transaction amount compared to their weekly average?

In [0]:
from pyspark.sql import functions as F

transactions_silver_tbl = spark.table("silver.transactions_silver")

user_weekly_avg_df = (
    transactions_silver_tbl
    .groupBy("client_id", "week_start")
    .agg(F.avg("amount").alias("weekly_avg_amount"))
)

user_spike_vs_weekly_avg_df = (
    transactions_silver_tbl.alias("t")
    .join(
        user_weekly_avg_df.alias("w"),
        on=["client_id", "week_start"],
        how="left"
    )
    .withColumn(
        "is_sharp_rise",
        F.when(F.col("amount") > F.col("weekly_avg_amount") * 2, True).otherwise(False)
    )
    .filter(F.col("is_sharp_rise") == True)
    .select(
        "client_id",
        "transaction_id",
        "transaction_date",
        "amount",
        "weekly_avg_amount",
        "is_sharp_rise"
    )
)

display(user_spike_vs_weekly_avg_df.limit(20))

client_id,transaction_id,transaction_date,amount,weekly_avg_amount,is_sharp_rise
957,7476973,2010-01-01,20.8100,-3.26600000,true
901,7477686,2010-01-01,58.1900,22.42000000,true
1379,7479631,2010-01-02,96.4800,45.42142857,true
1604,7481102,2010-01-02,84.0800,25.66411765,true
321,7482299,2010-01-02,42.8400,19.54333333,true
322,7486116,2010-01-03,77.2100,15.95555556,true
1204,7486776,2010-01-03,140.6300,46.98750000,true
1969,7490754,2010-01-04,73.9600,32.01382353,true
1967,7490790,2010-01-04,140.0000,34.49360000,true
1586,7491480,2010-01-05,120.0000,52.53000000,true


In [0]:
user_spike_vs_weekly_avg_df.write.mode("overwrite").saveAsTable("gold.user_spike_vs_weekly_avg")

In [0]:
display(spark.table("gold.user_spike_vs_weekly_avg").limit(10))

client_id,transaction_id,transaction_date,amount,weekly_avg_amount,is_sharp_rise
1484,7475385,2010-01-01,40.0000,18.22333333,true
135,7475607,2010-01-01,189.2900,44.17571429,true
1383,7478817,2010-01-01,128.5700,51.77285714,true
208,7481136,2010-01-02,96.7800,29.99000000,true
944,7483142,2010-01-03,243.6500,51.66222222,true
106,7484737,2010-01-03,106.3500,26.09937500,true
1759,7486621,2010-01-03,76.2100,19.83666667,true
1909,7491977,2010-01-05,113.1700,31.13729730,true
1687,7493286,2010-01-05,91.7100,36.58536585,true
1977,7494803,2010-01-05,459.0000,42.74720000,true


### Question 5: Which merchant categories exhibit the highest fraud rate?

In [0]:
from pyspark.sql import functions as F

transactions_silver_tbl = spark.table("silver.transactions_silver")

fraud_rate_by_mcc_df = (
    transactions_silver_tbl
    .groupBy("mcc", "merchant_category")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum(F.when(F.col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transactions")
    )
    .withColumn(
        "fraud_rate",
        F.col("fraud_transactions") / F.col("total_transactions")
    )
    .orderBy(F.desc("fraud_rate"), F.desc("fraud_transactions"))
)

display(fraud_rate_by_mcc_df.limit(10))

mcc,merchant_category,total_transactions,fraud_transactions,fraud_rate
4411,Cruise Lines,428,165,0.3855140186915888
5733,Music Stores - Musical Instruments,319,76,0.23824451410658307
3006,Miscellaneous Fabricated Metal Products,351,29,0.08262108262108261
5045,"Computers, Computer Peripheral Equipment",2793,204,0.07303974221267455
3144,Floor Covering Stores,334,23,0.0688622754491018
5732,Electronics Stores,6997,402,0.0574531942260969
3005,Miscellaneous Metal Fabrication,391,22,0.056265984654731455
3009,Fabricated Structural Metal Products,408,22,0.05392156862745098
5094,Precious Stones and Metals,5180,242,0.04671814671814672
3007,Coated and Laminated Products,381,17,0.04461942257217848


In [0]:
fraud_rate_by_mcc_df.write.mode("overwrite").saveAsTable("gold.fraud_rate_by_mcc")

In [0]:
display(spark.table("gold.fraud_rate_by_mcc").limit(10))

mcc,merchant_category,total_transactions,fraud_transactions,fraud_rate
4411,Cruise Lines,428,165,0.3855140186915888
5733,Music Stores - Musical Instruments,319,76,0.23824451410658307
3006,Miscellaneous Fabricated Metal Products,351,29,0.08262108262108261
5045,"Computers, Computer Peripheral Equipment",2793,204,0.07303974221267455
3144,Floor Covering Stores,334,23,0.0688622754491018
5732,Electronics Stores,6997,402,0.0574531942260969
3005,Miscellaneous Metal Fabrication,391,22,0.056265984654731455
3009,Fabricated Structural Metal Products,408,22,0.05392156862745098
5094,Precious Stones and Metals,5180,242,0.04671814671814672
3007,Coated and Laminated Products,381,17,0.04461942257217848


In [0]:
from pyspark.sql import functions as F

top_10_fraud_rate_by_mcc_df = (
    spark.table("gold.fraud_rate_by_mcc")
    .orderBy(F.desc("fraud_rate"), F.desc("fraud_transactions"))
    .limit(10)
)

top_10_fraud_rate_by_mcc_df.write \
    .mode("overwrite") \
    .saveAsTable("gold.top_10_fraud_rate_by_mcc")

### Question 6: Are there specific merchants with unusually high fraud volume?

In [0]:
from pyspark.sql import functions as F

transactions_silver_tbl = spark.table("silver.transactions_silver")

high_fraud_merchants_df = (
    transactions_silver_tbl
    .filter(F.col("is_fraud") == True)
    .groupBy("merchant_id", "merchant_city", "merchant_state", "merchant_category")
    .agg(
        F.count("*").alias("fraud_transaction_count"),
        F.sum("amount").alias("total_fraud_amount")
    )
    .orderBy(F.desc("fraud_transaction_count"), F.desc("total_fraud_amount"))
)

display(high_fraud_merchants_df.limit(10))

merchant_id,merchant_city,merchant_state,merchant_category,fraud_transaction_count,total_fraud_amount
60569,ONLINE,null,Wholesale Clubs,759,82309.6700
27092,ONLINE,null,Money Transfer,715,64747.0200
76639,ONLINE,null,Electronics Stores,284,43344.8800
32858,ONLINE,null,Department Stores,283,27824.9900
83018,Rome,Italy,Discount Stores,236,13363.7400
48919,Rome,Italy,Department Stores,221,18811.8100
99370,Rome,Italy,Department Stores,195,12686.2800
47399,ONLINE,null,"Digital Goods - Media, Books, Apps",176,8749.9200
34490,ONLINE,null,Miscellaneous Home Furnishing Stores,160,17468.5200
88260,Rome,Italy,"Grocery Stores, Supermarkets",149,5140.3600


In [0]:
high_fraud_merchants_df.write.mode("overwrite").saveAsTable("gold.high_fraud_merchants")

In [0]:
display(spark.table("gold.high_fraud_merchants").limit(10))

merchant_id,merchant_city,merchant_state,merchant_category,fraud_transaction_count,total_fraud_amount
60569,ONLINE,null,Wholesale Clubs,759,82309.6700
27092,ONLINE,null,Money Transfer,715,64747.0200
76639,ONLINE,null,Electronics Stores,284,43344.8800
32858,ONLINE,null,Department Stores,283,27824.9900
83018,Rome,Italy,Discount Stores,236,13363.7400
48919,Rome,Italy,Department Stores,221,18811.8100
99370,Rome,Italy,Department Stores,195,12686.2800
47399,ONLINE,null,"Digital Goods - Media, Books, Apps",176,8749.9200
34490,ONLINE,null,Miscellaneous Home Furnishing Stores,160,17468.5200
88260,Rome,Italy,"Grocery Stores, Supermarkets",149,5140.3600


In [0]:
from pyspark.sql import functions as F

top_10_fraud_amount_by_mcc_df = (
    spark.table("gold.fraud_amount_by_mcc")
    .orderBy(F.desc("total_fraud_amount"))
    .limit(10)
)

top_10_fraud_amount_by_mcc_df.write \
    .mode("overwrite") \
    .saveAsTable("gold.top_10_fraud_amount_by_mcc")

In [0]:
from pyspark.sql import functions as F

top_10_high_fraud_merchants_df = (
    spark.table("gold.high_fraud_merchants")
    .orderBy(F.desc("fraud_transaction_count"), F.desc("total_fraud_amount"))
    .limit(10)
)

top_10_high_fraud_merchants_df.write \
    .mode("overwrite") \
    .saveAsTable("gold.top_10_high_fraud_merchants")

### Question 7: How does fraud distribution vary by time of day (morning vs night)?

In [0]:
from pyspark.sql import functions as F

transactions_silver_tbl = spark.table("silver.transactions_silver")

fraud_by_time_of_day_df = (
    transactions_silver_tbl
    .groupBy("time_of_day")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum(F.when(F.col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transactions")
    )
    .withColumn(
        "fraud_rate",
        F.col("fraud_transactions") / F.col("total_transactions")
    )
    .orderBy(F.desc("fraud_transactions"))
)

display(fraud_by_time_of_day_df)

time_of_day,total_transactions,fraud_transactions,fraud_rate
Morning,5415684,5741,0.001060069235945081
Afternoon,4464677,5693,0.0012751202382613569
Evening,2260382,1525,6.746647248120008E-4
Night,1165172,373,3.201244108166005E-4


In [0]:
fraud_by_time_of_day_df.write.mode("overwrite").saveAsTable("gold.fraud_by_time_of_day")

In [0]:
display(spark.table("gold.fraud_by_time_of_day"))

time_of_day,total_transactions,fraud_transactions,fraud_rate
Morning,5415684,5741,0.001060069235945081
Afternoon,4464677,5693,0.0012751202382613569
Evening,2260382,1525,6.746647248120008E-4
Night,1165172,373,3.201244108166005E-4


### Question 8: What’s the average transaction amount for fraud vs non-fraud transactions?

In [0]:
from pyspark.sql import functions as F

transactions_silver_tbl = spark.table("silver.transactions_silver")

avg_amount_fraud_vs_nonfraud_df = (
    transactions_silver_tbl
    .groupBy("is_fraud")
    .agg(F.avg("amount").alias("avg_transaction_amount"))
    .orderBy(F.desc("is_fraud"))
)

display(avg_amount_fraud_vs_nonfraud_df)

is_fraud,avg_transaction_amount
true,110.23468197
false,42.90858094


In [0]:
avg_amount_fraud_vs_nonfraud_df.write.mode("overwrite").saveAsTable("gold.avg_amount_fraud_vs_nonfraud")

In [0]:
display(spark.table("gold.avg_amount_fraud_vs_nonfraud"))

is_fraud,avg_transaction_amount
true,110.23468197
false,42.90858094


### Question 9: Which merchant category has the highest total fraud amount?

In [0]:
from pyspark.sql import functions as F

transactions_silver_tbl = spark.table("silver.transactions_silver")

fraud_amount_by_mcc_df = (
    transactions_silver_tbl
    .filter(F.col("is_fraud") == True)
    .groupBy("mcc", "merchant_category")
    .agg(
        F.sum("amount").alias("total_fraud_amount"),
        F.count("*").alias("fraud_transaction_count")
    )
    .orderBy(F.desc("total_fraud_amount"))
)

display(fraud_amount_by_mcc_df.limit(10))

mcc,merchant_category,total_fraud_amount,fraud_transaction_count
5311,Department Stores,225647.1900,2251
4411,Cruise Lines,185946.7800,165
5300,Wholesale Clubs,113827.6500,991
5310,Discount Stores,81214.8900,859
4829,Money Transfer,66101.5200,725
5732,Electronics Stores,61171.3800,402
5712,"Furniture, Home Furnishings, and Equipment Stores",56989.4500,170
5719,Miscellaneous Home Furnishing Stores,34238.4500,313
4814,Telecommunication Services,33625.0400,162
5651,Family Clothing Stores,30051.5600,385


In [0]:
fraud_amount_by_mcc_df.write.mode("overwrite").saveAsTable("gold.fraud_amount_by_mcc")

In [0]:
display(spark.table("gold.fraud_amount_by_mcc").limit(10))

mcc,merchant_category,total_fraud_amount,fraud_transaction_count
5311,Department Stores,225647.1900,2251
4411,Cruise Lines,185946.7800,165
5300,Wholesale Clubs,113827.6500,991
5310,Discount Stores,81214.8900,859
4829,Money Transfer,66101.5200,725
5732,Electronics Stores,61171.3800,402
5712,"Furniture, Home Furnishings, and Equipment Stores",56989.4500,170
5719,Miscellaneous Home Furnishing Stores,34238.4500,313
4814,Telecommunication Services,33625.0400,162
5651,Family Clothing Stores,30051.5600,385


### Question 10: What are the total monetary losses due to fraud each day?

In [0]:
from pyspark.sql import functions as F

transactions_silver_tbl = spark.table("silver.transactions_silver")

daily_fraud_losses_df = (
    transactions_silver_tbl
    .filter(F.col("is_fraud") == True)
    .groupBy("transaction_date")
    .agg(
        F.sum("amount").alias("total_fraud_loss"),
        F.count("*").alias("fraud_transaction_count")
    )
    .orderBy("transaction_date")
)

display(daily_fraud_losses_df.limit(10))

transaction_date,total_fraud_loss,fraud_transaction_count
2010-01-01,0.1900,1
2010-01-03,339.0000,1
2010-01-04,11.6400,2
2010-01-05,8.7600,1
2010-01-07,-48.4600,2
2010-01-08,383.2400,4
2010-01-09,23.1000,1
2010-01-10,530.0100,5
2010-01-11,302.7000,2
2010-01-12,1014.4100,1


In [0]:
daily_fraud_losses_df.write.mode("overwrite").saveAsTable("gold.daily_fraud_losses")

In [0]:
display(spark.table("gold.daily_fraud_losses").limit(10))

transaction_date,total_fraud_loss,fraud_transaction_count
2010-01-01,0.1900,1
2010-01-03,339.0000,1
2010-01-04,11.6400,2
2010-01-05,8.7600,1
2010-01-07,-48.4600,2
2010-01-08,383.2400,4
2010-01-09,23.1000,1
2010-01-10,530.0100,5
2010-01-11,302.7000,2
2010-01-12,1014.4100,1


### Question 11: How many unique users commit fraudulent transactions per week?

In [0]:
from pyspark.sql import functions as F

transactions_silver_tbl = spark.table("silver.transactions_silver")

unique_fraud_users_weekly_df = (
    transactions_silver_tbl
    .filter(F.col("is_fraud") == True)
    .groupBy("week_start")
    .agg(
        F.countDistinct("client_id").alias("unique_fraud_users"),
        F.count("*").alias("fraud_transaction_count")
    )
    .orderBy("week_start")
)

display(unique_fraud_users_weekly_df.limit(10))

week_start,unique_fraud_users,fraud_transaction_count
2009-12-28,1,2
2010-01-04,5,15
2010-01-11,5,11
2010-01-18,7,18
2010-01-25,17,61
2010-02-01,16,70
2010-02-08,12,70
2010-02-15,13,57
2010-02-22,13,62
2010-03-01,10,65


In [0]:
unique_fraud_users_weekly_df.write.mode("overwrite").saveAsTable("gold.unique_fraud_users_weekly")

In [0]:
display(spark.table("gold.unique_fraud_users_weekly").limit(10))

week_start,unique_fraud_users,fraud_transaction_count
2009-12-28,1,2
2010-01-04,5,15
2010-01-11,5,11
2010-01-18,7,18
2010-01-25,17,61
2010-02-01,16,70
2010-02-08,12,70
2010-02-15,13,57
2010-02-22,13,62
2010-03-01,10,65


#### Question 12 Do fraud patterns show seasonal or monthly spikes?

In [0]:
from pyspark.sql import functions as F

transactions_silver_tbl = spark.table("silver.transactions_silver")

monthly_fraud_spikes_df = (
    transactions_silver_tbl
    .filter(F.col("is_fraud") == True)
    .groupBy("year_month")
    .agg(
        F.count("*").alias("fraud_transaction_count"),
        F.sum("amount").alias("total_fraud_amount"),
        F.countDistinct("client_id").alias("unique_fraud_users")
    )
    .orderBy("year_month")
)

display(monthly_fraud_spikes_df.limit(10))

year_month,fraud_transaction_count,total_fraud_amount,unique_fraud_users
2010-01,107,12253.5700,22
2010-02,259,32488.1100,40
2010-03,261,37454.2000,35
2010-04,237,29130.4200,34
2010-05,274,27258.8600,37
2010-06,182,15926.2900,31
2010-07,244,28099.7700,31
2010-08,229,24878.7500,35
2010-09,193,22063.8400,30
2010-10,224,29450.9800,30


In [0]:
monthly_fraud_spikes_df.write.mode("overwrite").saveAsTable("gold.monthly_fraud_spikes")

In [0]:
display(spark.table("gold.monthly_fraud_spikes").limit(10))

year_month,fraud_transaction_count,total_fraud_amount,unique_fraud_users
2010-01,107,12253.5700,22
2010-02,259,32488.1100,40
2010-03,261,37454.2000,35
2010-04,237,29130.4200,34
2010-05,274,27258.8600,37
2010-06,182,15926.2900,31
2010-07,244,28099.7700,31
2010-08,229,24878.7500,35
2010-09,193,22063.8400,30
2010-10,224,29450.9800,30


In [0]:
from pyspark.sql import functions as F

top_users_by_spike_count_df = (
    spark.table("gold.user_spike_vs_weekly_avg")
    .groupBy("client_id")
    .agg(F.count("*").alias("spike_transaction_count"))
    .orderBy(F.desc("spike_transaction_count"))
    .limit(10)
)

top_users_by_spike_count_df.write \
    .mode("overwrite") \
    .saveAsTable("gold.top_users_by_spike_count")

### Question 13: How has user behavior changed before versus after a fraudulent event?

In [0]:
from pyspark.sql import functions as F

transactions_silver_tbl = spark.table("silver.transactions_silver")

first_fraud_df = (
    transactions_silver_tbl
    .filter(F.col("is_fraud") == True)
    .groupBy("client_id")
    .agg(F.min("transaction_date").alias("first_fraud_date"))
)

user_behavior_before_after_fraud_df = (
    transactions_silver_tbl.alias("t")
    .join(first_fraud_df.alias("f"), on="client_id", how="inner")
    .withColumn(
        "behavior_period",
        F.when(F.col("transaction_date") < F.col("first_fraud_date"), "Before Fraud")
         .when(F.col("transaction_date") > F.col("first_fraud_date"), "After Fraud")
         .otherwise("Fraud Day")
    )
    .groupBy("client_id", "first_fraud_date", "behavior_period")
    .agg(
        F.count("*").alias("transaction_count"),
        F.avg("amount").alias("avg_transaction_amount"),
        F.sum("amount").alias("total_transaction_amount")
    )
    .orderBy("client_id", "behavior_period")
)

display(user_behavior_before_after_fraud_df.limit(20))

client_id,first_fraud_date,behavior_period,transaction_count,avg_transaction_amount,total_transaction_amount
0,2015-10-30,After Fraud,5240,49.50039313,259382.0600
0,2015-10-30,Before Fraud,7548,48.47069422,365856.8000
0,2015-10-30,Fraud Day,7,80.11571429,560.8100
1,2016-11-20,After Fraud,2815,35.74652931,100626.4800
1,2016-11-20,Before Fraud,7249,32.38834736,234783.1300
1,2016-11-20,Fraud Day,9,86.41777778,777.7600
2,2016-09-13,After Fraud,3428,26.81644107,91926.7600
2,2016-09-13,Before Fraud,7178,27.74794929,199174.7800
2,2016-09-13,Fraud Day,6,72.12166667,432.7300
3,2013-10-18,After Fraud,3863,48.21490551,186254.1800


In [0]:
user_behavior_before_after_fraud_df.write.mode("overwrite").saveAsTable("gold.user_behavior_before_after_fraud")

In [0]:
display(spark.table("gold.user_behavior_before_after_fraud").limit(10))

client_id,first_fraud_date,behavior_period,transaction_count,avg_transaction_amount,total_transaction_amount
0,2015-10-30,After Fraud,5240,49.50039313,259382.0600
0,2015-10-30,Before Fraud,7548,48.47069422,365856.8000
0,2015-10-30,Fraud Day,7,80.11571429,560.8100
1,2016-11-20,After Fraud,2815,35.74652931,100626.4800
1,2016-11-20,Before Fraud,7249,32.38834736,234783.1300
1,2016-11-20,Fraud Day,9,86.41777778,777.7600
2,2016-09-13,After Fraud,3428,26.81644107,91926.7600
2,2016-09-13,Before Fraud,7178,27.74794929,199174.7800
2,2016-09-13,Fraud Day,6,72.12166667,432.7300
3,2013-10-18,After Fraud,3863,48.21490551,186254.1800


### Question 14 Are fraudulent transactions more common on high-value purchases compared to low-value purchases?

In [0]:
from pyspark.sql import functions as F

transactions_silver_tbl = spark.table("silver.transactions_silver")

high_value_vs_low_value_fraud_df = (
    transactions_silver_tbl
    .groupBy("high_value_flag")
    .agg(
        F.count("*").alias("total_transactions"),
        F.sum(F.when(F.col("is_fraud") == True, 1).otherwise(0)).alias("fraud_transactions")
    )
    .withColumn(
        "fraud_rate",
        F.col("fraud_transactions") / F.col("total_transactions")
    )
    .orderBy(F.desc("high_value_flag"))
)

display(high_value_vs_low_value_fraud_df)

high_value_flag,total_transactions,fraud_transactions,fraud_rate
true,9185,100,0.010887316276537834
false,13296730,13232,9.951318858095186E-4


In [0]:
high_value_vs_low_value_fraud_df.write.mode("overwrite").saveAsTable("gold.high_value_vs_low_value_fraud")

In [0]:
display(spark.table("gold.high_value_vs_low_value_fraud"))

high_value_flag,total_transactions,fraud_transactions,fraud_rate
true,9185,100,0.010887316276537834
false,13296730,13232,9.951318858095186E-4
